# Exploratory Data Analysis

SQL-first validation of MetaPro RPKM sample data against taxonomy/pathway reference tables.

- [Design spec](../../../../docs/superpowers/specs/2026-06-11-exploratory-analysis-design.md)
- [Setup & workflow](../README.md)

In [1]:
from pathlib import Path

import duckdb

%load_ext sql

REPO_ROOT = Path("..").resolve().parents[1]  # notebooks → exploration → analytics → repo root
PARQUET_DIR = REPO_ROOT / "resources/db/parquet"
RPKM_FILES = [
    REPO_ROOT / "resources/example_data/test_rpkm_1.tsv",
    REPO_ROOT / "resources/example_data/test_rpkm_2.tsv",
]
TABLES = [
    "names", "nodes", "parents",
    "pathway_nodes", "pathway_edges",
    "pathway_superpathways", "superpathways",
]
KEY_COLS = ["GeneID", "Length", "Reads", "EC#", "RPKM", "Unclassified"]

conn = duckdb.connect()
%sql conn --alias duckdb

print(f"Repo root: {REPO_ROOT}")

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Repo root: /Users/sibyl/study/metapro-data-vis/.worktrees/exploration-eda


In [2]:
import sys

missing = []
for table in TABLES:
    p = PARQUET_DIR / f"{table}.parquet"
    if not p.exists():
        missing.append(str(p))
for rpkm in RPKM_FILES:
    if not rpkm.exists():
        missing.append(str(rpkm))

if missing:
    print("MISSING INPUT FILES:")
    for m in missing:
        print(f"  - {m}")
    print("\nRun: uv run python exploration/scripts/export_parquet.py")
    sys.exit(1)

for table in TABLES:
    p = PARQUET_DIR / f"{table}.parquet"
    n = conn.execute(f"SELECT COUNT(*) FROM '{p}'").fetchone()[0]
    mb = p.stat().st_size / (1024 * 1024)
    print(f"{table}: {n:,} rows ({mb:.1f} MB)")

for rpkm in RPKM_FILES:
    mb = rpkm.stat().st_size / (1024 * 1024)
    print(f"{rpkm.name}: {mb:.1f} MB")

print("All inputs present.")

names: 2,840,139 rows (137.3 MB)
nodes: 2,840,139 rows (10.8 MB)
parents: 2,840,134 rows (129.7 MB)
pathway_nodes: 23,864 rows (1.0 MB)
pathway_edges: 46,726 rows (2.5 MB)
pathway_superpathways: 193 rows (0.0 MB)
superpathways: 13 rows (0.0 MB)
test_rpkm_1.tsv: 51.0 MB
test_rpkm_2.tsv: 71.1 MB
All inputs present.
